In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
import sklearn.metrics as skmet
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:

df = pd.read_csv('/content/churn.csv')

In [ ]:
# Define the bins for age groups
bins = [10, 19, 29, 39, 49, 59, 120]  # The last value ensures all ages above 59 are grouped as '60+'

# Define the labels for age groups
labels = ['10-19', '20-29', '30-39', '40-49', '50-59', '60+']

# Use pd.cut() to create a new column with age groups
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

In [ ]:
df.shape

(36992, 25)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
class OutlierTrans(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=1.5):
        self.threshold = threshold

    def fit(self, X, y=None):
        # This transformer does not need any fitting, so we return self.
        return self

    def transform(self, X, y=None):
        X_new = X.copy()
        for col in X_new.columns:
            # Calculate the 1st and 3rd quartiles for the current column
            third_q, first_q = np.percentile(X_new[col], [75, 25])
            # Calculate the interquartile range
            interquartile_range = self.threshold * (third_q - first_q)
            # Define the outlier bounds
            outlier_high, outlier_low = third_q + interquartile_range, first_q - interquartile_range
            # Replace values outside the outlier bounds with NaN
            X_new.loc[(X_new[col] > outlier_high) | (X_new[col] < outlier_low), col] = np.nan
        return X_new.values

In [ ]:
class MakeOrdinal(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # This transformer does not need any fitting, so we return self.
        return self

    def transform(self, X, y=None):
        X_new = X.copy()
        for col in X_new.columns:
            # Get unique categories and sort them
            cats = np.sort(X_new[col].unique())
            # Replace categorical values with their ordinal values
            X_new[col] = X_new.apply(lambda x: int(np.where(cats == x[col])[0]), axis=1)
        return X_new.values

In [ ]:
class ReplaceVals(BaseEstimator, TransformerMixin):
    def __init__(self, repdict):
        self.repdict = repdict

    def fit(self, X, y=None):
        # This transformer does not need any fitting, so we return self.
        return self

    def transform(self, X, y=None):
        X_new = X.copy().replace(self.repdict)
        return X_new.values

In [ ]:
# Numerical columns for analysis
num_cols = [
    'days_since_last_login', 'avg_time_spent', 'avg_transaction_value',
    'points_in_wallet', 'avg_frequency_login_days'
]

# Special columns that may require special handling
special_col1 = ['age_group']
special_col2 = ['membership_category']

# Categorical columns for analysis
cat_cols = ['gender',
    'medium_of_operation', 'internet_option', 'feedback', 'complaint_status'
]

# Binary columns (having two categories) for analysis
binary_cols = [
    'joined_through_referral', 'used_special_discount',
    'offer_application_preference', 'past_complaint'
]

# Replacement dictionary for 'membership_category'
rep_dict = {
    'membership_category': {
        'No Membership': 0,
        'Basic Membership': 1,
        'Silver Membership': 2,
        'Gold Membership': 3,
        'Premium Membership': 4,
        'Platinum Membership': 5
    }
}

In [ ]:
# Numerical columns for analysis
num_cols = ['days_since_last_login', 'avg_time_spent', 'avg_transaction_value',
            'points_in_wallet', 'avg_frequency_login_days']

# Special columns that may require special handling
special_col1 = ['age']
special_col2 = ['complaint_status']
special_col3 = ['membership_category']

# Categorical columns for analysis
cat_cols = ['gender', 'region_category', 'preferred_offer_types',
            'medium_of_operation', 'internet_option', 'feedback']

# Binary columns (having two categories) for analysis
binary_cols = ['joined_through_referral', 'used_special_discount',
               'offer_application_preference', 'past_complaint']

# Target column for analysis
target_col = ['churn_risk_score']

# Columns not in use for the analysis
column_not_in_use = ['security_no', 'referral_id', 'joining_date', 'last_visit_time', 'Unnamed: 0']

In [ ]:
from sklearn.model_selection import train_test_split

# Define the feature columns
feature_cols = num_cols + binary_cols + cat_cols + special_col1 + special_col2

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols],
    df[['churn_risk_score']],
    test_size=0.2,
    random_state=0
)

In [ ]:
ohe = OneHotEncoder(drop='first', sparse=False)
standtrans = make_pipeline(
    OutlierTrans(3),
    SimpleImputer(strategy="median"),
    StandardScaler()
)
spectrans1 = make_pipeline(
    MakeOrdinal(),
    StandardScaler()
)
spectrans2 = make_pipeline(
    ReplaceVals(rep_dict),
    StandardScaler()
)
spectrans3 = make_pipeline(
    ReplaceVals(rep_dict)
)
bintrans = make_pipeline(ohe)
cattrans = make_pipeline(ohe)

coltrans = ColumnTransformer(
    transformers=[
        ("stand", standtrans, num_cols),
        ("spec1", spectrans1, special_col1),
        ("spec2", spectrans2, special_col2),
        ("bin", bintrans, binary_cols),
        ("cat", cattrans, cat_cols),
    ]
)

In [ ]:
# Create a DecisionTreeClassifier
dtc_example = DecisionTreeClassifier(min_samples_leaf=5, max_depth=2)

pipe0 = make_pipeline(coltrans, dtc_example)

# Fit the pipeline to your training data
pipe0.fit(X_train, y_train.values.ravel())

ValueError: A given column is not a column of the dataframe

In [ ]:
# Get binary column names from the pipeline
new_binary_cols = pipe0.named_steps['columntransformer'] \
    .named_transformers_['bin'] \
    .named_steps['onehotencoder'] \
    .get_feature_names_out(input_features=binary_cols)

# Get categorical column names from the pipeline
new_cat_cols = pipe0.named_steps['columntransformer'] \
    .named_transformers_['cat'] \
    .named_steps['onehotencoder'] \
    .get_feature_names_out(input_features=cat_cols)

# Combine all column names into a single array
new_cols = np.concatenate((
    np.array(num_cols + special_col1 + special_col2),
    new_binary_cols,
    new_cat_cols
))

AttributeError: 'ColumnTransformer' object has no attribute 'transformers_'

In [ ]:
# Get feature importances from the trained DecisionTreeClassifier
feature_imp = pipe0.named_steps['decisiontreeclassifier'].tree_.compute_feature_importances(normalize=False)

# Create a mask to filter features with importances greater than 0
feature_impgt0 = feature_imp > 0

# Stack feature importances with corresponding feature names
feature_implabs = np.column_stack((feature_imp.ravel(), new_cols))


AttributeError: 'DecisionTreeClassifier' object has no attribute 'tree_'

In [ ]:
pred = pipe0.predict(X_test)
accuracy = skmet.accuracy_score(y_test.values.ravel(), pred)
sensitivity = skmet.recall_score(y_test.values.ravel(), pred)
specificity = skmet.recall_score(y_test.values.ravel(), pred, pos_label=0)
precision = skmet.precision_score(y_test.values.ravel(), pred)

print("accuracy: %.2f, sensitivity: %.2f, specificity: %.2f, precision: %.2f" % (accuracy, sensitivity, specificity, precision))

AttributeError: 'ColumnTransformer' object has no attribute 'transformers_'

We should explore alternative values for these hyperparameters that will yield a better-performing model. Let’s do a randomized grid search to find those values.

let’s set up a pipeline with the column transformations, the oversampling, and a decision tree classifier. We will also create a dictionary with ranges for the minimum leaf size and maximum tree depth hyperparameters. Note that there are two underscores after decisiontreeclassifier for each dictionary key:

In [ ]:
dtc = DecisionTreeClassifier(random_state=0)
pipe1 = make_pipeline(coltrans, dtc)

In [ ]:
from scipy.stats import randint

# Define parameter grid for DecisionTreeClassifier
dtc_params = {
    'decisiontreeclassifier__min_samples_leaf': randint(100, 1200),
    'decisiontreeclassifier__max_depth': randint(2, 11)
}

In [ ]:
rs = RandomizedSearchCV(pipe1, dtc_params, cv=5,
n_iter=20, scoring="roc_auc")
rs.fit(X_train, y_train.values.ravel())

NameError: name 'pipe1' is not defined

In [ ]:
rs.best_params_

In [ ]:
rs.best_score_

In [ ]:
# Make predictions using the random search model
pred2 = rs.predict(X_test)

# Calculate the confusion matrix
cm = skmet.confusion_matrix(y_test, pred2)

# Create a ConfusionMatrixDisplay object
cmplot = skmet.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])

# Plot the confusion matrix
cmplot.plot()

# Set the title and axis labels
cmplot.ax_.set(
    title='Churn Prediction Confusion Matrix',
    xlabel='Predicted Value',
    ylabel='Actual Value'
)

NameError: name 'rs' is not defined

Let’s look at the accuracy, sensitivity, specificity, and precision scores to see if there is much improvement in the metrics from the model without hyperparameter tuning.


In [ ]:
# Calculate the metrics
accuracy = skmet.accuracy_score(y_test.values.ravel(), pred2)
sensitivity = skmet.recall_score(y_test.values.ravel(), pred2)
specificity = skmet.recall_score(y_test.values.ravel(), pred2, pos_label=0)
precision = skmet.precision_score(y_test.values.ravel(), pred2)

# Print the metrics
print(f"accuracy: {accuracy:.2f}, sensitivity: {sensitivity:.2f}, specificity: {specificity:.2f}, precision: {precision:.2f}")

NameError: name 'pred2' is not defined